# 01 — Data Understanding and Exploration

Document the dataset context, structure, quality, distributions, and initial analytical findings before data preparation and modeling.

## 1. Study Context

Customer churn occurs when a customer ends their relationship with a service provider. In the telecommunications context, identifying patterns associated with churn can support more focused customer-retention actions.

This study uses the Telco Customer Churn dataset to investigate how customer characteristics, subscribed services, contract conditions, tenure, and charges relate to the `Churn` outcome. The exploratory stage will evaluate the dataset's structure and quality, identify relevant behavioral patterns, and establish evidence for the later development and comparison of binary classification models.

The dataset source summarizes the business objective as:

> “Predict behavior to retain customers. You can analyze all relevant customer data and develop focused customer retention programs.” — IBM Sample Data Sets

The study is guided by the following questions:

* Which customer, service, and account characteristics are most associated with churn?
* Are there data-quality issues or class imbalance that may affect the analysis?
* Which findings should guide data preparation, feature treatment, and model evaluation?

## 2. Dataset Source

The study uses the **Telco Customer Churn** dataset published on Kaggle by **BlastChar** and attributed to **IBM Sample Data Sets**.

| Item                     | Value                                                                                                          |
| ------------------------ | -------------------------------------------------------------------------------------------------------------- |
| Dataset                  | Telco Customer Churn                                                                                           |
| Platform                 | Kaggle                                                                                                         |
| Kaggle handle            | `blastchar/telco-customer-churn`                                                                               |
| Original attribution     | IBM Sample Data Sets                                                                                           |
| Acquisition method       | `kagglehub`, through `scripts/download_data.py`                                                                |
| Local raw-data directory | `data/raw/telco-customer-churn/`                                                                               |
| Retrieved version        | Latest available version at execution time                                                                     |
| Usage note               | Preserve the original source attribution and review the source usage conditions before redistributing the data |

The downloaded files are treated as immutable source data. Cleaning, type correction, filtering, and other transformations must not overwrite files under `data/raw/`. Derived datasets should be written to `data/interim/` or `data/processed/`.

In [13]:
from scripts.project_context import get_project_context


# Cria o contexto do projeto atual.
PROJECT = get_project_context()

# Mantém o caminho absoluto disponível para operações internas.
PROJECT_ROOT = PROJECT.root

# Monta o caminho da pasta de dados brutos de forma multiplataforma.
RAW_DATA_DIR = PROJECT.path(
    "data",
    "raw",
    "telco-customer-churn",
)

print(f"Project: {PROJECT.name}")
print(f"Raw data directory: {PROJECT.display(RAW_DATA_DIR)}")

Project: dataset-study-telco-customer-churn
Raw data directory: data/raw/telco-customer-churn


In [17]:
# Importa a função de aquisição de datasets hospedados no Kaggle.
from scripts.download_data import acquire_kaggle_dataset


# Identificador público do dataset no Kaggle.
DATASET_HANDLE = "blastchar/telco-customer-churn"

# Garante que o dataset esteja disponível no diretório de dados brutos.
acquisition = acquire_kaggle_dataset(
    handle=DATASET_HANDLE,
    destination=RAW_DATA_DIR,
)

RAW_DATA_DIR = acquisition.destination

print(f"Dataset source: Kaggle — {acquisition.source_reference}")
print(f"Raw data directory: {PROJECT.display(RAW_DATA_DIR)}")
print("Dataset files:")

for file_path in acquisition.files:
    print(f"- {file_path.name}")

Dataset source: Kaggle — blastchar/telco-customer-churn
Raw data directory: data/raw/telco-customer-churn
Dataset files:
- WA_Fn-UseC_-Telco-Customer-Churn.csv


In [18]:
import pandas as pd


DATASET_FILE = acquisition.require_one_file("*.csv")
df = pd.read_csv(DATASET_FILE)

print(f"Dataset file: {DATASET_FILE.name}")
print(f"Loaded shape: {df.shape}")

df.head()

Dataset file: WA_Fn-UseC_-Telco-Customer-Churn.csv
Loaded shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 3. Unit of Observation

Each row represents one telecommunications customer account observed at the time the dataset snapshot was produced.

The dataset combines information from different aspects of the customer relationship, including:

* customer profile and household characteristics;
* subscribed telephone and internet services;
* contract, billing, and payment conditions;
* customer tenure and accumulated charges;
* the churn outcome recorded for the account.

The analytical unit is therefore the **customer account**, identified by `customerID`. It is not an individual service subscription, transaction, invoice, support interaction, or repeated observation over time.

Because the dataset represents a customer-level snapshot, variables such as `tenure`, `MonthlyCharges`, and `TotalCharges` summarize the state of the customer relationship at the observation point. The `Churn` column indicates whether the customer left the company according to the outcome captured in the dataset.

Before continuing with the analysis, the uniqueness and completeness of `customerID` must be verified to confirm that each row corresponds to one distinct customer account.

In [19]:
# Column that identifies the analytical unit represented by each row.
OBSERVATION_ID = "customerID"

if OBSERVATION_ID not in df.columns:
    raise KeyError(
        f"Observation identifier not found: {OBSERVATION_ID}"
    )

print(f"Unit of observation: customer account")
print(f"Observation identifier: {OBSERVATION_ID}")
print(f"Number of rows: {len(df):,}")

Unit of observation: customer account
Observation identifier: customerID
Number of rows: 7,043


In [20]:
# Validate whether each row represents one distinct customer account.
observation_summary = {
    "rows": len(df),
    "non_null_ids": df[OBSERVATION_ID].notna().sum(),
    "unique_ids": df[OBSERVATION_ID].nunique(dropna=True),
    "missing_ids": df[OBSERVATION_ID].isna().sum(),
    "duplicated_rows_by_id": df[OBSERVATION_ID].duplicated(
        keep=False
    ).sum(),
}

for label, value in observation_summary.items():
    print(f"{label.replace('_', ' ').title()}: {value:,}")

Rows: 7,043
Non Null Ids: 7,043
Unique Ids: 7,043
Missing Ids: 0
Duplicated Rows By Id: 0


In [23]:
# Inspect repeated identifiers if the granularity validation fails.
duplicated_observations = (
    df.loc[
        df[OBSERVATION_ID].duplicated(keep=False)
    ]
    .sort_values(OBSERVATION_ID)
)

duplicated_observations

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn


In [24]:
# Stop the analysis if the expected customer-level granularity is violated.
if observation_summary["missing_ids"] > 0:
    raise ValueError(
        f"{OBSERVATION_ID} contains "
        f"{observation_summary['missing_ids']} missing values."
    )

if observation_summary["unique_ids"] != observation_summary["rows"]:
    raise ValueError(
        "The number of unique customer identifiers does not match "
        "the number of rows. The dataset may contain repeated "
        "customer accounts."
    )

print(
    "Validation passed: each row represents one distinct "
    "customer account."
)

Validation passed: each row represents one distinct customer account.


## 4. Dataset Structure

Summarize the number of rows and columns, the available fields, and the overall organization of the data.

## 5. Feature and Target Identification

Identify the input features, the target variable, identifier fields, and columns that require special treatment.

## 6. Data Types

Review the expected and observed data types, highlighting columns that may have been loaded with an incorrect type.

## 7. Data Dictionary

Document the meaning, expected values, units, and analytical role of each relevant column.

## 8. Missing and Invalid Values

Investigate absent, blank, inconsistent, or invalid values and describe their potential impact on the analysis.

## 9. Duplicate Records

Check for duplicated observations and determine whether they represent data-quality issues or valid repeated records.

## 10. Target Distribution

Examine how the target classes or values are distributed and identify possible imbalance or unusual patterns.

## 11. Numerical Feature Exploration

Explore distributions, ranges, central tendencies, variability, outliers, and unusual values in numerical features.

## 12. Categorical Feature Exploration

Explore category frequencies, rare values, cardinality, inconsistent labels, and potentially relevant groupings.

## 13. Feature Relationships

Investigate associations between features to identify redundancy, dependencies, interactions, or unexpected relationships.

## 14. Feature-to-Target Relationships

Compare each relevant feature with the target to identify patterns, hypotheses, and variables that may support modeling.

## 15. Potential Data Leakage

Identify fields or transformations that may reveal the target directly or contain information unavailable at inference time.

## 16. Initial Data-Quality Findings

Consolidate the main structural and quality issues that must be addressed during data preparation.

## 17. Key Exploratory Insights

Summarize the most relevant patterns, contrasts, and hypotheses discovered during the exploratory analysis.

## 18. Preparation Decisions

Record the preliminary decisions that should guide cleaning, transformation, feature engineering, and dataset splitting.

## 19. Next Steps

List the actions that will be continued in the data-preparation and model-selection stages.